# FLAN-T5 Opinion Summarization
---

Summarizing multiple opinions/comments using FLAN-T5-large model.

**Loading the Model**

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5-large model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype=torch.float32  # CPU usage
)

**Defining the Summarization Function**

In [5]:
def create_prompt(text, chunks=False) -> str:
    if chunks:
        return (
            "The following are summaries of different groups of comments:\n"
            f"{text}\n\n"
            "Instruction: Combine these into one coherent summary that captures "
            "the overall sentiment and key themes."
        )
    else: 
        return (
            "Analyze the following comments:\n"
            f"{text}\n\n"
            "Instruction: Provide a concise summary of the overall sentiment "
            "and the main themes expressed in these comments."
        )

def chunk_comments(comments, tokenizer, max_tokens=400) -> list[str]: 
    """
    Chunk comments into smaller groups based on token count; <400, leaving space for the prompt
    """
    chunks, chunk, tokens = [], [], 0

    for c in comments:
        text = f"- {c}"
        len_t = len(tokenizer.encode(text))

        if chunk and tokens + len_t > max_tokens:
            chunks.append("\n".join(chunk))
            chunk, tokens = [], 0

        chunk.append(text)
        tokens += len_t

    return chunks + (["\n".join(chunk)] if chunk else [])


def generate_summary(prompt, tokenizer, model, max_length=128, min_length=20, num_beams=4) -> str:
    """
    Generate a summary for the given prompt using the provided tokenizer and model.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    ids = model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        do_sample=False,
        early_stopping=True
    )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


In [6]:
def summarize_opinions(comments, tokenizer, model) -> dict[str, int] | dict[str, str, int]:
    """
    Summarize a list of comments using hierarchical summarization if needed.
    Returns either a dictionary of (single summary, token count) or a dictionary of (intermediate summaries, final summary, token_count)
    """
    combined_text = "\n".join(f"- {c}" for c in comments)

    # single-pass summarization is possible
    token_count = len(tokenizer.encode(combined_text))
    if token_count <= 400:
        return {
            "summary": generate_summary(create_prompt(combined_text), tokenizer, model),
            "token_count": token_count
        }

    # Hierarchical summarization
    chunks = chunk_comments(comments, tokenizer)
    chunk_summaries = [
        generate_summary(create_prompt(chunk), tokenizer, model)
        for chunk in chunks
    ]

    combined_summaries = "\n".join(f"- {s}" for s in chunk_summaries)
    return {
        "intermediate_summaries": combined_summaries,
        "final_summary": generate_summary(create_prompt(combined_summaries, chunks=True), tokenizer, model),
        "token_count": token_count
    }


### Usage

In [ ]:
import pandas as pd

df = pd.read_csv("sustainability_social_media_posts.csv")

# head (mixed)
# sample_comments = df["post_text"].head(10).tolist()

# filtered (same sentiment and topic)
sample_comments = df[(df["post_sentiment"] == "Positive") & 
                       (df["climate_topic"] == "Climate Policy")]["post_text"].tolist()

print("RAW COMMENTS:\n")
for comment in sample_comments[:10]:
    print(f"{comment}")
print("\n")

result = summarize_opinions(
    sample_comments[:10],
    tokenizer, 
    model
)
print(f"Total comments to summarize: {len(sample_comments)}")
print("Token count for input to model:", result["token_count"])

if result["token_count"] <= 400:
    print("Summary Type: Single-pass summarization\n")
    print(f"\nSUMMARY:\n{result['summary']}")
else:
    print("Summary Type: Hierarchical summarization. Chunking...\n")
    print(f"INTERMEDIATE SUMMARIES:\n{result['intermediate_summaries']}\n")
    print(f"\nFINAL SUMMARY:\n{result['final_summary']}")

RAW COMMENTS:

Renewable energy requires change through activism. Global warming should be embraced through education. Climate mitigation prevents catastrophic warming scenarios.
Environmental protection is everyone's responsibility at every level. Reforestation is urgent through activism. Water conservation reduces pollution starting today.
Water conservation can save our planet starting today. Green innovation drives economic growth while protecting environment. Climate activism promotes cleaner air to empower communities.
Green infrastructure projects provide multiple environmental benefits. Sustainable transportation reduces urban pollution significantly. Circular economy builds a greener future by reducing waste.
LED lighting saves energy consumption dramatically. Renewable energy is everyone's responsibility by reducing waste. Green buildings improve energy efficiency markedly.
Green technology is essential through innovation. Climate justice should be embraced with collective ef

## Fine Tuning
---

## Model Evaluation
---